# Demo 5 · Base vs your fine-tune vs Jev

**Part 06 · Benchmarking and evaluation** · about 10 minutes

Three deciders, one interface, the same 48 fresh test cases (tickets, hard tickets, and three
domains the model never saw):

* **base** Qwen3, scored by Exercise 3 (or here, if that file is missing)
* **fine-tuned**: your model from the Hub, the local `outputs/model`, or the instructor's prepared checkpoint
* **Jev**: live with a key; otherwise saved answers if `data/replay/jev_fresh_test.jsonl` exists

Every decider gets the same review rule: top probability below 0.8 → review.
The predictions are saved to `outputs/predictions_demo5.jsonl` for Exercise 5.

In [ ]:
%pip install -q -r requirements-finetune.txt

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import json, time
import pandas as pd
from dmw import scorer as S, datagen, metrics, jev

BASE_MODEL = "Qwen/Qwen3-0.6B"
MY_MODEL = f"{config.key('HF_USERNAME')}/qwen3-0.6b-decision-router"   # your Hub repo
PREPARED_CHECKPOINT = ""            # instructor's backup model ID, if you have no model yet
MODE = config.mode("auto")
cases = datagen.load_test("data/fresh_test.json")
print("MODE =", MODE, "|", len(cases), "cases | device:", config.device())

## 1 · Score the three deciders

In [ ]:
preds = []

def add(model_name, rows):
    for r in rows:
        preds.append({"model": model_name, "id": r["id"], "group": r["group"], "expected": r["expected"],
                      "choice": r["choice"], "p_top": max(r["probabilities"].values()),
                      "review": max(r["probabilities"].values()) < 0.8 or "orderings_disagree" in r.get("flags", []),
                      "latency_ms": r.get("latency_ms")})

# base
base_file = config.OUTPUTS / "baseline_base_model.json"
if base_file.exists() and json.load(open(base_file))["model"] == BASE_MODEL:
    add("base", json.load(open(base_file))["rows"])
else:
    m, t = S.load(BASE_MODEL); add("base", S.score_cases(S.LetterScorer(m, t), cases)); del m

# fine-tuned: Hub repo, then local folder, then the prepared checkpoint
for candidate in [MY_MODEL, str(config.OUTPUTS / "model"), PREPARED_CHECKPOINT]:
    if not candidate or candidate.startswith("None/"):
        continue
    try:
        m, t = S.load(candidate); FINETUNED = candidate; break
    except Exception as e:
        print("could not load", candidate, "→", type(e).__name__)
else:
    raise RuntimeError("No fine-tuned model found: run notebook 07 or set PREPARED_CHECKPOINT.")
add("fine-tuned", S.score_cases(S.LetterScorer(m, t), cases))
print("fine-tuned model:", FINETUNED)

In [ ]:
# Jev on the same cases: one Choice per case, criteria = the option descriptions
def jev_rows():
    saved = config.REPLAY / "jev_fresh_test.jsonl"
    if MODE == "replay" and saved.exists():
        return [json.loads(l) for l in saved.read_text().splitlines()]
    if MODE != "live":
        return []
    rows = []
    for c in cases:
        out = jev.ask(c["context"], {"d": jev.question("choice", c["question"], c["choices"])})
        a = out["answers"]["d"]
        rows.append({"id": c["id"], "group": c["group"], "expected": c["expected"], "choice": a["value"],
                     "probabilities": a["probabilities"], "latency_ms": out["latency_ms"]})
    return rows

jr = jev_rows()
if jr:
    add("jev", jr)
else:
    print("No Jev answers: add TYPESAFE_API_KEY for live mode (cached after the first run).")
P = pd.DataFrame(preds)
P.to_json(config.OUTPUTS / "predictions_demo5.jsonl", orient="records", lines=True)
P.groupby("model").size()

## 2 · Accuracy by group

In [ ]:
acc = P.assign(correct=P.choice == P.expected).pivot_table(index="group", columns="model", values="correct", aggfunc="mean")
acc.loc["overall"] = P.assign(correct=P.choice == P.expected).groupby("model").correct.mean()
(acc * 100).round(1)

In [ ]:
rows = []
for mname, g in P.groupby("model"):
    nf = g[~g.review]
    rows.append({"model": mname, "accuracy": metrics.accuracy(g.expected.tolist(), g.choice.tolist()),
                 "macro_f1": metrics.macro_f1(g.expected.tolist(), g.choice.tolist()),
                 "review rate": g.review.mean(),
                 "accuracy when not flagged": metrics.accuracy(nf.expected.tolist(), nf.choice.tolist()) if len(nf) else None,
                 "errors not flagged": int(((nf.choice != nf.expected)).sum()),
                 **metrics.latency(g.latency_ms.tolist())})
pd.DataFrame(rows).set_index("model").round(3)

## 3 · Confusion matrix of the fine-tuned model (support tickets)

In [ ]:
ft = P[(P.model == "fine-tuned") & P.group.str.startswith("tickets")]
metrics.confusion(ft.expected.tolist(), ft.choice.tolist())

## 4 · Disagreements: read them, decide who is right

In [ ]:
wide = P.pivot(index="id", columns="model", values="choice")
text = {c["id"]: c["text"] for c in cases}; gold = {c["id"]: c["expected"] for c in cases}
wide.insert(0, "gold", wide.index.map(gold)); wide.insert(0, "text", wide.index.map(text))
others = [m for m in wide.columns if m not in ("text", "gold")]
dis = wide[wide[others].nunique(axis=1) > 1]
print(len(dis), "cases where the deciders disagree")
dis

## 5 · Optional: off-task check on AG News

Your model was trained to route tickets. How does it do on news topics, compared with the saved
Kev-4B, Laya and Jev runs on the same 100 examples? Needs internet to download the texts.

In [ ]:
RUN_AG_NEWS = False
if RUN_AG_NEWS:
    from dmw import bench
    ag = bench.attach_texts("ag_news", bench.frame("ag_news"))
    labs = bench.labels("ag_news"); sc = S.LetterScorer(m, t)
    mine = [sc.score(x, list(labs), descriptions=labs, question="Select the main news topic.").choice for x in ag.text]
    tbl = bench.table("ag_news")
    tbl.loc["your fine-tune"] = {"n": len(ag), "accuracy": round(metrics.accuracy(ag.gold.tolist(), mine), 3),
                                 "macro_f1": round(metrics.macro_f1(ag.gold.tolist(), mine), 3), "failures": 0}
    display(tbl)